In [0]:
import urllib.request
from datetime import datetime
from dateutil.relativedelta import relativedelta

target_date = datetime.now() - relativedelta(months=7)
year_month = target_date.strftime('%Y-%m')
filename = f"yellow_tripdata_{year_month}.parquet"
url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{filename}"

volume_path = f"/Volumes/nyc_taxi_etl/default/raw_landing/{filename}"

try:
    print(f"Streaming {filename} directly to Azure Storage...")
    
    urllib.request.urlretrieve(url, volume_path)
    
    print(f"Success! Raw file saved directly to: {volume_path}")

except Exception as e:
    print(f"Pipeline failed: {e}")

Streaming yellow_tripdata_2025-10.parquet directly to Azure Storage...
Success! Raw file saved directly to: /Volumes/nyc_taxi_etl/default/raw_landing/yellow_tripdata_2025-10.parquet


In [0]:
%fs ls 'abfss://nyctaxidata@storagenyctaxietl.dfs.core.windows.net/'

In [0]:
%sql
create external location if not exists nyctaxidata_demo
  url 'abfss://nyctaxidata@storagenyctaxietl.dfs.core.windows.net/'
  with (storage credential sc_nyc_taxi_etl)

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS ext_bronze_data
  URL 'abfss://bronzedata@storagenyctaxietl.dfs.core.windows.net/'
  with (storage credential sc_nyc_taxi_etl)

In [0]:
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import current_timestamp

target_date = datetime.now() - relativedelta(months=5)
year_month = target_date.strftime('%Y-%m')
filename = f"yellow_tripdata_{year_month}.parquet"

volume_path = f"/Volumes/nyc_taxi_etl/default/raw_landing/{filename}"
bronze_table_path = "abfss://bronzedata@storagenyctaxietl.dfs.core.windows.net/yellow_tripdata"
bronze_table_name = "nyc_taxi_etl.default.bronze_yellow_tripdata"

try:
    print(f"1. Reading raw Parquet file into Spark from Volume...")
    raw_df = spark.read.parquet(volume_path)
    
    bronze_df = raw_df.withColumn("ingestion_timestamp", current_timestamp())
    
    print(f"3. Saving to Bronze External Delta Table: {bronze_table_name}...")
    
    bronze_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .option("path", bronze_table_path) \
        .saveAsTable(bronze_table_name)
        
    print(f"Success! Inserted {bronze_df.count()} rows into the Bronze table.")

except Exception as e:
    print(f"Bronze layer processing failed: {e}")

1. Reading raw Parquet file into Spark from Volume...
3. Saving to Bronze External Delta Table: nyc_taxi_etl.default.bronze_yellow_tripdata...
Success! Inserted 4428699 rows into the Bronze table.


In [0]:
total_rows = spark.read.table("nyc_taxi_etl.default.gold_hourly_metrics").count()

print(f"Total rows: {total_rows}")

Total rows: 747


In [0]:
total_rows = spark.read.table("nyc_taxi_etl.default.bronze_yellow_tripdata").count()

print(f"Total rows: {total_rows}")

Total rows: 13286097
